# topology2.py — Full Walkthrough

This notebook explains every part of `topology2.py` from the Intertwined Lattices project.

## What is this file about?

`symmetry.py` (covered in a separate notebook) handles the raw 3-D geometry of a unit cell — reading it, classifying nodes, finding geometric symmetries.

`topology2.py` builds **on top** of that to answer a much harder question:

> Given a lattice unit cell that has `N` parallel strands running through every edge, in how many *topologically distinct* ways can you route those strands through the structure?

In physical terms: imagine `N` fibres (or wires, or filaments) weaving through a 3-D lattice. This file enumerates all distinct **topological routings** — i.e., all distinct ways to connect ports on one face of the unit cell to ports on the opposite face, passing through junctions. It also detects which routings form **closed loops** versus **open (periodic) strands**.

## High-level pipeline

```
.dat file
  │
  ▼
Topology(filename, N)          ← builds the Graph, sets up all data
  │
  ▼
topology.generate()            ← find all valid, unique, conjugate path-collections
  │  ├─ generate_solutions()   ← the combinatorial core
  │  └─ instantiate_strands()  ← assign strand IDs to each path
  │
  ▼
topology.compute_loops()       ← trace which strands close into loops
  │
  ▼
topology.plot_solution_v1/v2() ← visualise a specific routing
```

## 1 — Imports and Key Concepts

### Libraries used

| Library | Purpose |
|---|---|
| `collections.defaultdict` | Auto-initialising dictionary for building adjacency structures |
| `collections.Counter` | Counting how many times each edge appears across a collection of paths |
| `collections.deque` | Efficient stack/queue for BFS and DFS traversal |
| `itertools.combinations`, `permutations`, `product` | Generating all possible path orderings and strand assignments |
| `numpy` | Numerical arrays, vector math |
| `networkx` | Graph structures (used via import from `symmetry`) |
| `matplotlib` | All 2-D and 3-D plotting |
| `matplotlib.animation` | Exporting animated GIFs/MP4s of solution sequences |
| `math` | Factorial (used in combinatorial cardinality calculations) |
| `pickle` | Saving/loading `Topology` instances to disk |
| `time` | Progress timing during long searches |
| `copy` | Deep-copying data structures during backtracking |
| `symmetry` (local) | Imports everything from `symmetry.py` — `read_lattice_file`, `Graph`, `find_symmetries`, etc. |
| `plotting` (local) | Additional project-specific plotting helpers |

### Key vocabulary

| Term | Meaning |
|---|---|
| **Path** | An ordered sequence of edge IDs that forms a valid route through the lattice (e.g., `(0, 4, 2)` means traverse edges 0, then 4, then 2) |
| **Solution** | A collection of paths with multiplicities that together cover every edge in the lattice exactly `N` times |
| **Conjugate paths** | Two paths that enter/exit through paired port/counterport nodes — required for physical periodicity |
| **Strand** | A single physical filament; `N` strands run through each edge. A strand has an ID (0 to N-1) on each edge it crosses |
| **Permuted solution** | A specific assignment of strand IDs to the paths in a solution |
| **Loop** | A strand that closes back on itself when followed through the tiled lattice |
| **Open strand** | A strand that continues forever periodically without closing |

## 2 — Path Generation: `all_unique_sequences()`

### What is a "path" in this context?

Every edge in the lattice is identified by an integer ID.  
A **path** is an ordered sequence of edge IDs that represents a continuous route a strand can take — for example `(0, 4, 2)` means the strand travels along edge 0, then edge 4, then edge 2 in sequence.

The paths must be physically valid:
- The edges in the sequence must be **contiguous** (share a node).
- The path must start and end at **port** nodes (the boundary nodes of the unit cell).
- A path **cannot** revisit the same junction node.

`all_unique_sequences()` generates the raw **candidate set** of all edge orderings — the valid ones are filtered later.

---

### Symmetry reduction: treating reverses as identical

Since a strand going `A → B → C` is physically the same as one going `C → B → A`, the function keeps only the **lexicographically smaller** of each sequence and its reverse.

---

### Inputs

| Parameter | Type | Description |
|---|---|---|
| `elements` | set or list | The edge IDs to combine — typically `self.edges.keys()` |
| `min_length` | int | Minimum path length in edges (default 2) |
| `max_length` | int | Maximum path length in edges (default = total number of edges) |

### Output

A **sorted list of tuples** — every unique ordered subset of edge IDs (up to reversal), sorted first by length then lexicographically.

```
Input edges: {0, 1, 2, 3}
Example outputs: (0,1), (0,2), (1,3), (0,1,2), (1,2,3), ...
```

In [1]:
from collections import defaultdict, Counter, deque
from itertools import chain, combinations, permutations, combinations_with_replacement, product
import copy, math, pickle, random, time
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

def all_unique_sequences(elements, min_length=2, max_length=None):
    elements = list(elements)
    sequences = set()
    if max_length is None:
        max_length = len(elements)
    for subset_size in range(min_length, max_length + 1):
        for subset in combinations(elements, subset_size):
            for perm in permutations(subset):
                sequences.add(min(perm, perm[::-1]))
    return sorted(sequences, key=lambda x: (len(x), x))


# ── Demo with 4 edges ──
edges_demo = {0, 1, 2, 3}
seqs = all_unique_sequences(edges_demo, min_length=2, max_length=3)

print(f"Total candidate sequences (length 2–3) from 4 edges: {len(seqs)}")
print("\nAll sequences:")
for s in seqs:
    print(f"  {s}")

Total candidate sequences (length 2–3) from 4 edges: 18

All sequences:
  (0, 1)
  (0, 2)
  (0, 3)
  (1, 2)
  (1, 3)
  (2, 3)
  (0, 1, 2)
  (0, 1, 3)
  (0, 2, 1)
  (0, 2, 3)
  (0, 3, 1)
  (0, 3, 2)
  (1, 0, 2)
  (1, 0, 3)
  (1, 2, 3)
  (1, 3, 2)
  (2, 0, 3)
  (2, 1, 3)


## 3 — Coverage Condition: `condition_fn()`

### The core constraint — "every edge must be covered exactly N times"

Once you have a collection of paths, you need to check that together they use every edge of the lattice **exactly `N` times** (once per strand).  
This is the `condition_fn` — it is the primary filter that separates valid solutions from invalid ones.

**Example:**  
If N=2, and the lattice has edges {0, 1, 2, 3}:
- Collection: `{(0,1): 1, (2,3): 1, (0,2): 1, (1,3): 1}` — each of edges 0,1,2,3 appears exactly 2 times total ✓
- Collection: `{(0,1): 2, (2,3): 1}` — edges 0 and 1 appear 2 times, but 2 and 3 only once ✗

---

### Inputs

| Parameter | Type | Description |
|---|---|---|
| `collection` | `Counter {tuple: int}` | A mapping of paths to their multiplicities |
| `values` | list | All edge IDs that must be covered (typically all edges) |
| `N` | int | Required coverage count per edge |

### Output

`True` if every edge appears exactly `N` times across the collection, `False` otherwise.

In [2]:
def condition_fn(collection, values, N):
    """Check if each value appears exactly N times across all sequences in the collection."""
    total_counts = Counter({value: 0 for value in values})
    for seq, mult in collection.items():
        for value in seq:
            total_counts[value] += mult
    return all(count == N for count in total_counts.values())


# ── Demo ──
all_edges = [0, 1, 2, 3]
N = 2

# Valid: every edge covered exactly 2 times
valid_collection = Counter({(0, 1): 1, (2, 3): 1, (0, 2): 1, (1, 3): 1})
# Invalid: edge 2 and 3 are only covered once
invalid_collection = Counter({(0, 1): 2, (2, 3): 1})

print(f"Valid collection:   {dict(valid_collection)}")
print(f"  → condition_fn = {condition_fn(valid_collection, all_edges, N)}")

print(f"\nInvalid collection: {dict(invalid_collection)}")
print(f"  → condition_fn = {condition_fn(invalid_collection, all_edges, N)}")

Valid collection:   {(0, 1): 1, (2, 3): 1, (0, 2): 1, (1, 3): 1}
  → condition_fn = True

Invalid collection: {(0, 1): 2, (2, 3): 1}
  → condition_fn = False


## 4 — Conjugacy: `are_conjugate_paths()` and `is_conjugate()`

### What is conjugacy?

When you tile the unit cell infinitely in all directions, a strand exiting through one face must re-enter through the **opposite** face.
The port on one face and the counterport on the opposite face are **periodic twins** — they are connected by the lattice repeat vector.

**Conjugacy** is the constraint that enforces this physical requirement at the level of paths:

> Two paths are **conjugate** if one exits through a port on a given face at the same relative position that the other enters through the paired counterport on the opposite face.

In other words, conjugate paths are the two halves of the same physical strand crossing the unit cell boundary.

A **collection is conjugate** if every path in it has exactly one conjugate partner (or is self-conjugate — it connects a port directly to its own counterport).

---

### `are_conjugate_paths(path1, path2, edges, counterports, conjugacy_type)`

Checks if two specific paths are conjugate by examining whether their starting/ending edges share port/counterport node pairs.

**Inputs**: two paths (tuples of edge IDs), the edge dict, the counterport dict, and `conjugacy_type` (only type 0 is implemented)  
**Output**: `True` or `False`

---

### `is_conjugate(set_of_paths, edges, counterports, conjugacy_type)`

Checks an entire **collection** of paths: iterates through all paths and tries to pair each one with a conjugate partner. If any path is left unpaired, returns `False`.

**Inputs**: a frozenset of `(path, multiplicity)` pairs, plus the edge/counterport dicts  
**Output**: `True` if the full collection is conjugate, `False` otherwise

## 5 — Multiplicity Search: `generate_valid_multiplicities()` and `generate_multiplicity_collections()`

### The combinatorial problem

Given a list of valid paths, you need to assign a **multiplicity** (how many times that path is used) to each one, such that:
1. Every edge is covered exactly `N` times (`condition_fn`).
2. The collection is conjugate (`is_conjugate`).
3. No path is used more than `max_multiplicity` times.

This is an **exact cover** type problem and can be very expensive computationally.

---

### `generate_valid_multiplicities(sequences, max_multiplicity_edge, values, max_multiplicity_path)`

A **recursive backtracking generator** that yields tuples of multiplicities.

It processes paths one at a time and prunes branches early: if assigning multiplicity `m` to the current path would already over-fill some edge (exceed `N` uses), that branch is skipped immediately.

**Inputs**:
- `sequences`: list of paths (each a tuple of edge IDs)
- `max_multiplicity_edge`: the required total per edge = N
- `values`: all edge IDs
- `max_multiplicity_path`: cap on how many times any single path can be used

**Output**: a **generator** — each yield is a tuple like `(0, 1, 0, 2, 1, ...)` giving the multiplicity of each path

---

### `generate_multiplicity_collections(sequences, values, max_multiplicity, condition_fn, ...)`

The **outer search loop** — calls `generate_valid_multiplicities` exhaustively (or with the fast/random `discard_threshold` mode), applies `condition_fn` and conjugacy checks, and accumulates valid collections.

**Inputs**:
- `sequences`: all valid paths
- `values`: all edge IDs
- `max_multiplicity`: N (strands per edge)
- `condition_fn`: the coverage test
- `stop_early`: stop after this many valid collections found (or `False` for exhaustive)
- `discard_threshold` / `restart`: switches to a randomised fast search mode
- `edges`, `counterports`, `conjugacy_type`: needed for conjugacy checks

**Output**: a `set` of `frozenset` objects — each frozenset is a valid collection of `(path_tuple, multiplicity)` pairs

In [3]:
def generate_valid_multiplicities(sequences, max_multiplicity_edge, values, max_multiplicity_path=None):
    if max_multiplicity_path is None:
        max_multiplicity_path = max_multiplicity_edge
    num_elements = len(sequences)
    element_counts = {v: 0 for v in values}

    def backtrack(index, current_multiplicity, element_counts):
        if index == num_elements:
            if all(count == max_multiplicity_edge for count in element_counts.values()):
                yield tuple(current_multiplicity)
            return
        sequence_values = set(sequences[index])
        max_allowed = max_multiplicity_path
        for v in sequence_values:
            max_allowed = min(max_allowed, max_multiplicity_edge - element_counts[v])
        for m in range(max_allowed + 1):
            new_element_counts = element_counts.copy()
            for v in sequence_values:
                new_element_counts[v] += m
            yield from backtrack(index + 1, current_multiplicity + [m], new_element_counts)

    yield from backtrack(0, [], element_counts)


# ── Demo: 4 edges, N=2, paths of length 2 only ──
demo_sequences = [(0, 1), (0, 2), (1, 3), (2, 3), (0, 3), (1, 2)]
demo_values = [0, 1, 2, 3]
N = 2

results = list(generate_valid_multiplicities(demo_sequences, N, demo_values))
print(f"Generated {len(results)} multiplicity tuples that cover each edge exactly {N} times")
print(f"\nSequences:  {demo_sequences}")
print(f"\nValid multiplicities (one per sequence position):")
for r in results:
    active = [(seq, m) for seq, m in zip(demo_sequences, r) if m > 0]
    print(f"  {r}  →  active paths: {active}")

Generated 6 multiplicity tuples that cover each edge exactly 2 times

Sequences:  [(0, 1), (0, 2), (1, 3), (2, 3), (0, 3), (1, 2)]

Valid multiplicities (one per sequence position):
  (0, 0, 0, 0, 2, 2)  →  active paths: [((0, 3), 2), ((1, 2), 2)]
  (0, 1, 1, 0, 1, 1)  →  active paths: [((0, 2), 1), ((1, 3), 1), ((0, 3), 1), ((1, 2), 1)]
  (0, 2, 2, 0, 0, 0)  →  active paths: [((0, 2), 2), ((1, 3), 2)]
  (1, 0, 0, 1, 1, 1)  →  active paths: [((0, 1), 1), ((2, 3), 1), ((0, 3), 1), ((1, 2), 1)]
  (1, 1, 1, 1, 0, 0)  →  active paths: [((0, 1), 1), ((0, 2), 1), ((1, 3), 1), ((2, 3), 1)]
  (2, 0, 0, 2, 0, 0)  →  active paths: [((0, 1), 2), ((2, 3), 2)]


## 6 — The `Graph` Class (topology2 version)

`topology2.py` defines its **own version** of `Graph` (separate from the one in `symmetry.py`), with one key difference in `generate_counteredges()`:

In `symmetry.py`, a counteredge is found by looking up the counterport of any port node in an edge.  
In `topology2.py`, the counteredge search is **more detailed**: it explicitly iterates over junction nodes to find the matching edge on the opposite face, and handles **suppressed ports** (ports that are not connected to anything on the opposite face) by assigning `counteredge = -1`.

This is important because suppressed ports mark boundaries of the lattice where no periodicity is expected — for example in a lattice that is periodic in only 2 of 3 dimensions.

### Attributes after construction

| Attribute | Description |
|---|---|
| `points`, `connections`, `types`, `status` | Raw data from `read_lattice_file()` |
| `ports` | Indices of all port and counterport nodes |
| `junctions` | Indices of all junction nodes |
| `edges` | `{edge_id: (node_a, node_b)}` |
| `counterports` | `{port_idx: counterport_idx}` bidirectional |
| `counteredges` | `{edge_id: twin_edge_id}` — `-1` if twin is suppressed |
| `translations` | `{edge_id: 3D vector}` — how far to shift to reach the twin edge |
| `symmetries` | All distance-preserving node permutations (from `find_symmetries()`) |

### Additional method: `plot_lattice()`

This version of Graph adds `plot_lattice()` (3-D rendering with edge midpoint labels) in addition to `plot_graph()` (NetworkX spring layout). Edge IDs are annotated at midpoints in purple, making it easy to identify which integer corresponds to which strut.

## 7 — The `Topology` Class: `__init__` and Stored Attributes

`Topology` is the central object that ties everything together. You create one like this:

```python
topo = Topology('lattice_BCC.dat', N=2)
```

### Inputs to `__init__`

| Parameter | Type | Meaning |
|---|---|---|
| `filename` | `str` | Path to the `.dat` lattice file |
| `N` | `int` | Number of strands (routing layers) |
| `conjugacy` | `bool` | Whether to filter solutions by conjugacy (default `True`) |
| `power_set_min_length` | `int` | Minimum loop length to keep (default `1`) |
| `power_set_max_length` | `int` | Maximum loop length to keep (default `None` = unlimited) |

### What `__init__` does

1. Reads the `.dat` file via `read_lattice_file()`.
2. Builds a `Graph` object (the topology2 version).
3. Extracts `ports`, `junctions`, `edges`, `counterports`, `counteredges`, `translations`, `symmetries` directly from the Graph.
4. Stores `N`, `conjugacy`, and the power-set length limits.
5. Initialises all result containers to `None` (they are filled lazily by `generate()`).

### Key attributes available after construction

| Attribute | What it holds |
|---|---|
| `self.graph` | The `Graph` object built from the `.dat` file |
| `self.edges` | `{edge_id: (node_a, node_b)}` |
| `self.counteredges` | `{edge_id: twin}` (`-1` if suppressed) |
| `self.translations` | `{edge_id: 3D shift vector}` |
| `self.symmetries` | List of node-index permutations representing lattice symmetries |
| `self.N` | Number of strands |
| `self.max_multiplicity` | Max times any single edge can appear in a routing path (equals `N`) |
| `self.solutions` | `None` until `generate()` runs — then a list of valid solutions |
| `self.permuted_solutions` | All strand-labelled variants of every solution |
| `self.expanded_solutions` | Same, but also broken into loops and open strands |
| `self.loops` | Loop structures per solution, filled by `compute_loops()` |

## 8 — `generate()` and `generate_solutions()`: the Main Pipeline

Calling `topo.generate()` is **the one method you need to call** to produce all valid routing solutions. Internally it chains together all the helper functions described in earlier sections.

### `generate()` — high-level orchestrator

**Inputs:** nothing (uses current `self` state)  
**Outputs:** populates `self.solutions`, `self.permuted_solutions`, `self.expanded_solutions`, `self.loops`

Step-by-step what it does:

1. Calls `generate_solutions()` → stores result in `self.solutions`
2. Calls `instantiate_strands()` → stores result in `self.permuted_solutions`
3. Calls `compute_loops()` → stores result in `self.loops` and `self.expanded_solutions`

---

### `generate_solutions()` — the core search

This is where the combinatorial search happens. It finds every distinct way to route `N` strands through every edge exactly the right number of times.

**Inputs:** `(self)` — reads `self.N`, `self.edges`, `self.counteredges`, `self.symmetries`, `self.conjugacy`

**Outputs:** a list of *solutions*, where each solution is a list of `N` *paths*

#### Pipeline inside `generate_solutions()`:

| Step | Function used | What it does |
|---|---|---|
| 1. Enumerate all possible paths | `all_unique_sequences()` | Generates all acyclic sequences of edges that a single strand could follow |
| 2. Filter feasible paths | `condition_fn()` | Keeps only paths where each edge pair `(e, counteredge(e))` appears at most once |
| 3. Compute valid edge-use counts | `generate_valid_multiplicities()` | For each combination of N paths, checks that together they use every edge exactly N times |
| 4. Build solutions from multiplicities | `generate_multiplicity_collections()` | Groups paths into N-tuples that satisfy the coverage constraint |
| 5. Remove conjugate duplicates | `is_conjugate()` + `are_conjugate_paths()` | If two solutions are related by a lattice symmetry, keep only one |

**Output format:**  
Each solution is a list of N paths such as:
```
[[0, 1, 3, 5], [2, 4, 6, 7], ...]
```
where integers are edge IDs. The `counteredge` of the last edge in one path connects back to the first edge of the same or another path, forming a valid traversal of the whole lattice.

---

### What is a "solution"?

A solution defines **which edges each strand passes through** (the topology), but NOT the order in which strands are labelled on each edge. That comes next in `instantiate_strands()`.

## 9 — `instantiate_strands()`, `_v1()`, `_v2()`: Assigning Strand Labels

A *solution* tells you which edges each strand group uses, but does not say **which specific strand (0, 1, 2, …)** sits in which slot on each edge. `instantiate_strands()` enumerates all the distinct ways to label each edge-slot with a strand number.

---

### The concept

Imagine an edge that is used by 3 strands. On that edge there are 3 slots. The strands could sit in those slots in 3! = 6 different orders. But many of those orders are equivalent because they can be mapped to each other by the same global permutation of strand labels. `instantiate_strands` finds all **non-equivalent** label assignments.

---

### Three variants

| Method | Strategy | Use case |
|---|---|---|
| `instantiate_strands()` | Index into a precomputed permutation list using modular arithmetic | Fast, deterministic — default |
| `instantiate_strands_v1()` | Recursive backtracking | Reference implementation, slow but easy to verify |
| `instantiate_strands_v2()` | Iterative DFS (stack-based backtracking) | Same result as v1, avoids Python recursion limit |

---

### Inputs (all three variants)

| Parameter | Meaning |
|---|---|
| `solution` | One solution from `self.solutions` — a list of N paths |
| `N` | Number of strands |

### Output

A list of **permuted solutions**, each being a list of N paths with **integer strand labels 0…N-1** at every edge position.

```
# Example for N=2, one solution with 4 edges:
# solution = [[e0, e1], [e2, e3]]
# permuted solutions:
#   strand 0 follows [e0, e1], strand 1 follows [e2, e3]
#   strand 0 follows [e2, e3], strand 1 follows [e0, e1]
```

The total number of permuted solutions per base solution is bounded by `compute_cardinality_strands_permutations()` (see next section).

---

### How the default `instantiate_strands()` works

1. Computes `M = compute_cardinality_strands_permutations(solution)`.
2. For each index `i` in `0..M-1`, derives a unique assignment by factoring `i` through the factorials of each edge's multiplicity.
3. Applies `apply_permutation()` to re-label paths according to that assignment.
4. Calls `find_unique_sets()` to deduplicate any assignments that turn out to be identical after symmetry reduction.

This avoids explicitly generating all permutations upfront, making it efficient even for large `N`.

## 10 — `compute_cardinality_strands_permutations()`: Counting Distinct Assignments

Before actually generating all permuted solutions, this method tells you **how many distinct strand assignments exist** for each base solution.

### Formula

For a solution with `N` strands and a set of edges where path $i$ appears with multiplicity $m_i$:

$$\text{cardinality} = \frac{(N!)^{|\text{edges}|}}{\prod_i m_i!}$$

- $(N!)^{|\text{edges}|}$ counts all ways to assign strand labels at every edge.
- Dividing by $\prod m_i!$ removes assignments that are equivalent due to repeated paths.

### Inputs

| Parameter | Meaning |
|---|---|
| `solution` | (Optional) Index of a specific solution. If `None`, computes for all solutions. |

### Output

A dictionary `{solution_index: cardinality_integer}`. For example:
```
{0: 12, 1: 6, 2: 4, ...}
```

### Why it's useful

- Tells you the size of the search space before calling `instantiate_strands()`.
- A large cardinality means many distinct strand weavings exist for that base solution.
- A cardinality of 1 means there is only one way to label strands — that solution has no strand-permutation freedom.

## 11 — `compute_loops()`: Tracing Strands Across the Infinite Lattice

Once strands have been labelled on each edge, `compute_loops()` asks: *if you follow a strand continuously through the periodic lattice, does it ever come back to where it started (a loop), or does it extend to infinity (an open strand)?*

### The concept

The lattice is periodic. Each edge $e$ has a paired `counteredge` on the opposite face of the unit cell and a `translation` vector that tells you how far you have moved into the neighbouring unit cell. 

Starting from a path in the base unit cell, the method follows the strand by:
1. Looking at the last edge $e$ of the current path.
2. Finding `counteredge(e)` — the entry edge in the neighbouring unit cell.
3. Finding which path in the permuted solution starts with `counteredge(e)` **and** has the same strand ID.
4. Accumulating the `translation` vector each time you jump to a neighbour.
5. Stopping when either:
   - The accumulated translation is zero and you're back at a path you've already visited → **loop**.
   - The accumulated translation is nonzero when you revisit → **open strand** (extends to infinity).

### Inputs

| Parameter | Meaning |
|---|---|
| `solution` | Index into `self.solutions` |
| `permuted_solution` | Index into `self.permuted_solutions[solution]` |
| `threshold` | Maximum displacement before declaring a strand open (default = 10× max edge length) |

### Outputs (stored in `self.*`)

| Attribute | Contents |
|---|---|
| `self.loops[(sol, perm)]` | List of loops — each loop is a list of path IDs from the permuted solution that form the loop |
| `self.expanded_solutions[(sol, perm)]` | All strand trajectories as lists of `(path, unit_cell_offset)` pairs |
| `self.expanded_loops[(sol, perm)]` | Trajectories for strands that are loops |
| `self.expanded_open_strands[(sol, perm)]` | Trajectories for strands that are open |

### What "loop" vs "open strand" means physically

- **Loop**: the strand forms a closed circuit through the lattice — it passes through a finite, repeating set of unit cells and returns to its starting point.
- **Open strand**: the strand never returns — it travels infinitely far in some direction, like a helix or a straight rod.

## 12 — Plotting Methods

`topology2.py` provides several methods to visualise solutions as 2-D schematics. These help you see at a glance how strands cross over and under each other.

---

### `plot_solution_v1(solution, permutation, loops, starting_seed, color_map)`

The **high-quality, publication-ready** plotter. Uses Bézier curve rendering.

| Parameter | Meaning |
|---|---|
| `solution` | Index into `self.solutions` |
| `permutation` | Index into `self.permuted_solutions[solution]`. Pass `None` to draw the port-level solution (no sub-node splitting) |
| `loops` | `True` to draw loops as closed curves with crossing highlights |
| `starting_seed` | Random seed for colour generation (for reproducible colours) |
| `color_map` | `'conjugate'` (pairs conjugate paths with the same colour), `'black'`, `'color'`, or `None` |

**What it draws:**
- Ports are placed on a unit circle; junctions at the centre or on an inner ring.
- If `permutation` is given, each port is split into `N` sub-nodes arranged tangentially — one per strand slot.
- Each strand path is drawn as a Bézier arc connecting its sub-nodes through the junction.
- Crossing points between strands are detected and the curves are rendered with over-under relationships.

---

### `plot_solution_v2(solution, permutation, loops, starting_seed, ax, color_map)`

A **subplot-compatible** version of `plot_solution_v1`. Accepts an optional `ax` argument so it can be embedded in a grid of subplots. Used internally by `plot_multiple_solutions()`. Lower visual fidelity than v1 but faster to render.

---

### `plot_multiple_solutions(solutions, permutations, loops, nrows, ncols, figsize, color_map)`

Draws a grid of `plot_solution_v2` panels — one per solution/permutation pair.

| Parameter | Meaning |
|---|---|
| `solutions` | List of solution indices (default: all solutions) |
| `permutations` | Matching list of permutation indices (or `None` for each) |
| `nrows`, `ncols` | Grid layout (auto-computed if not given) |

---

### `save_solution_animation(filename, solutions, permutations, loops, interval, dpi)`

Generates an animated GIF or MP4 by calling `plot_solution_v1` for each `(solution, permutation)` frame and saving with matplotlib's `FuncAnimation`.

| Parameter | Meaning |
|---|---|
| `filename` | Output path (`.gif` or `.mp4`) |
| `interval` | Milliseconds between frames |
| `dpi` | Image resolution |

---

### `compute_color()` (standalone helper)

Used internally by all plotters. Assigns a colour to each path based on:
- `'conjugate'` mode: conjugate path pairs share one colour, the other pair gets another.
- `'black'` mode: everything is black.
- `'color'` mode: a new random colour per path, seeded for reproducibility.

## 13 — `compute_directional_indicator_vector()` and `compute_connectivity_matrix()`

These methods convert the abstract topology into a compact numerical description of which faces of the unit cell are connected to which.

---

### `compute_directional_indicator_vector(port_idx)`

**Input:** the index of a port node in the graph.

**Output:** a 6-element numpy array $v(p)$.

The unit cell has 6 faces: $-x, +x, -y, +y, -z, +z$ (indices 0–5). The vector $v(p)$ describes which face the port $p$ sits on:

$$v_k(p) = \frac{\max(0,\ \pm \text{coordinate})}{\|p\|_1}$$

where the sign follows the face direction and $\|\cdot\|_1$ is the L1 norm of the port position. Each element of $v(p)$ is in $[0, 1]$ and the vector encodes the fractional position of the port on each face.

For example, a port right on the $+x$ face gives $v = [0, 1, 0, 0, 0, 0]$.

---

### `compute_connectivity_matrix(port_level_realization_index)`

**Input:** index of a solution in `self.solutions` (default `0`).

**Output:** a $6 \times 6$ numpy array $C$ — the **Directional Connectivity Matrix**.

$C_{kl}$ counts how many paths connect face $k$ to face $l$ (symmetrised):

$$C_{kl} = \sum_\text{path} m_\text{path} \Big[ v_k(p_\text{start}) v_l(p_\text{end}) + v_l(p_\text{start}) v_k(p_\text{end}) - \delta_{kl}\, v_k(p_\text{start}) v_l(p_\text{end}) \Big]$$

where $m_\text{path}$ is the path multiplicity.

**What it tells you physically:** which pairs of faces are connected by strands, and how many strands connect them. This is used to characterise the mechanical or electrical connectivity of the metamaterial unit cell in each direction.

---

### `compute_loop_ratio(port_level_realization_index, permutation_index)`

A simple derived metric: $\text{ratio} = \text{number of loops} / \text{total number of strands}$. A ratio of 0 means all strands are open; a ratio of 1 means all strands close into loops.

## 14 — `save()`, `load()`, and `export_topology()`

Running `generate()` on a large lattice with many strands can take a long time. These three methods let you persist and share results.

---

### `save(filename)` / `Topology.load(filename)`

Uses Python's `pickle` module to serialise the **entire `Topology` object** (all solutions, permuted solutions, loops, graph, etc.) to a binary file.

```python
topology.save('topology_bcc_N4.pickle')          # Save
topology = Topology.load('topology_bcc_N4.pickle')  # Load back
```

**Format:** binary pickle file.  
**Caution:** pickle files are Python-version and class-definition sensitive. If you rename or restructure the `Topology` class, old pickles may fail to load.

---

### `export_topology(filename)`

Writes a **human-readable text file** with:

1. **Node positions** — `index: x, y, z` for every node.
2. **Symmetries** — each symmetry as a list of node index mappings.
3. **Edges** — `edge_id: node1, node2`.
4. **Port-level solutions** — each solution as `P<n>:` followed by lines of `multiplicity, edge_id1, edge_id2, ...`.
5. **Strand-level permutations** — each permuted solution as `P<n>S<m>:` followed by lines of `edge_id, strand_id` pairs.

**Typical usage:**
```python
topology.export_topology('./Dataset/BCC_N4.txt')
```

**When to use `export_topology` vs `save`:**
- Use `save` when you want to load the object back into Python and continue working with it.
- Use `export_topology` when you want a portable, language-agnostic record of results that can be read by other tools or collaborators.

## 15 — The `__main__` Block: Running the Full Pipeline

The bottom of `topology2.py` contains a `if __name__ == '__main__':` block that shows a complete end-to-end use of the Topology class. Here is a plain-English walkthrough of what it does:

```python
# 1. Create a Topology for the BCC lattice with 4 strands
topology = Topology('lattice_BCC.dat', N=4, conjugacy=True)
```
Reads the unit cell file, builds the Graph, and sets up all attributes. `conjugacy=True` means solutions that are mirror images (or rotations) of each other under the lattice symmetry will be deduplicated.

---

```python
# 2. Run the full search pipeline
topology.generate(stop_early=False, discard_threshold=None, restart=1, permuted_solutions='all')
```
- `stop_early=False` → keep searching even after many solutions are found.
- `discard_threshold=None` → do not prune partial solutions early.
- `restart=1` → run the search from scratch (do not resume a previous run).
- `permuted_solutions='all'` → enumerate every strand assignment for every solution.

After this call, `topology.solutions`, `topology.permuted_solutions`, and `topology.loops` are all populated.

---

```python
# 3. Export the results to a text file
topology.export_topology('./Dataset/BCC_N4.txt')
```

Writes the full catalogue of solutions and strand permutations to a human-readable file for later use.

---

```python
# 4. Visualise one specific solution
topology.plot_solution_v2(solution=18, permutation=3533, loops=False, starting_seed=1000, color_map='color')
plt.show()
```

Draws the 2-D schematic of solution index 18, strand permutation 3533, using a multi-colour scheme.

---

### Key optional parameters of `generate()`

| Parameter | Effect |
|---|---|
| `stop_early=True` | Stop after finding the first valid set of solutions (faster but incomplete) |
| `discard_threshold=N` | Discard partial path collections that are already too far from being valid (prunes search tree, faster) |
| `restart=K` | Restart the generator `K` times with different random seeds to find more solutions |
| `permuted_solutions='all'` | Compute all strand assignments (slow for large N); `'none'` skips this step |

## Summary: topology2.py at a Glance

### Full pipeline from `.dat` file to exported dataset

```
lattice_BCC.dat
       │
       ▼
  read_lattice_file()           ← symmetry.py
       │  nodes, edges, types
       ▼
  Graph (topology2 version)
       │  .edges, .counteredges, .translations, .symmetries
       ▼
  Topology.__init__(N=4)
       │
       ├── generate_solutions()
       │       │
       │       ├── all_unique_sequences()     → all edge-sequence candidates
       │       ├── condition_fn()             → filter: each edge-pair used ≤ N times
       │       ├── generate_valid_multiplicities()
       │       │       └── generate_multiplicity_collections()  → N-path sets covering all edges
       │       ├── is_conjugate()             → each path must have a conjugate partner
       │       └── find_unique_sets()         → remove symmetry-equivalent duplicates
       │             → self.solutions  (port-level)
       │
       ├── instantiate_strands()              → strand 0…N-1 labelled on every edge
       │             → self.permuted_solutions
       │
       ├── compute_loops()                    → follow strands across periodic boundary
       │             → self.loops
       │             → self.expanded_solutions
       │
       ├── compute_connectivity_matrix()      → 6×6 face-to-face connectivity C
       │
       ├── plot_solution_v1/v2()              → 2-D strand schematic (Bézier curves)
       ├── plot_multiple_solutions()          → subplot grid of schematics
       ├── save_solution_animation()          → animated GIF/MP4
       │
       ├── save() / load()                    → pickle serialisation
       └── export_topology()                  → human-readable text dataset
```

### Key concepts recap

| Concept | One-line definition |
|---|---|
| **Path** | An ordered sequence of edge IDs that a single strand follows through the unit cell |
| **Solution** | A set of N paths (with multiplicities) that together cover every edge exactly N times |
| **Conjugate paths** | Two paths whose entry/exit ports are counterports of each other |
| **Permuted solution** | A solution with strand numbers 0…N-1 explicitly assigned to each edge slot |
| **Loop** | A strand that closes on itself when traced through the periodic lattice |
| **Open strand** | A strand that extends to infinity (like a helix or rod) |
| **Connectivity matrix C** | 6×6 matrix encoding which faces of the unit cell are connected by how many strands |